# Fig S2D: Pattern metrics

In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
from scipy.spatial import distance

def extract_raw_lattice_data(image_paths, scale=0.0125, bridge_gap=2, eps_factor=0.04):
    master_records = []

    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None: continue
        
        overlay = img.copy()
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, np.array([130, 40, 40]), np.array([180, 255, 255]))
        
        kernel = np.ones((3,3), np.uint8)
        mask_closed = cv2.dilate(mask, kernel, iterations=bridge_gap)
        mask_closed = cv2.morphologyEx(mask_closed, cv2.MORPH_CLOSE, kernel)
        
        inverted = cv2.bitwise_not(mask_closed)
        contours, _ = cv2.findContours(inverted, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        
        img_units = []
        centroids = []

        for cnt in contours:
            area_px = cv2.contourArea(cnt)
            # Filter background and micro-noise
            if 400 < area_px < 15000:
                peri_px = cv2.arcLength(cnt, True)
                
                # Metric calculation
                q_value = (4 * np.pi * area_px) / (peri_px**2) if peri_px > 0 else 0
                approx = cv2.approxPolyDP(cnt, eps_factor * peri_px, True)
                
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    cx, cy = int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])
                    centroids.append((cx, cy))
                    
                    img_units.append({
                        'Source': os.path.basename(img_path),
                        'Area_mm2': area_px * (scale**2),
                        'Perimeter_mm': peri_px * scale,
                        'Q_Factor': q_value,
                        'Centroid_X': cx,
                        'Centroid_Y': cy
                    })
                    
                    # Verification Overlay
                    cv2.drawContours(overlay, [approx], -1, (0, 255, 0), 2)

        df_temp = pd.DataFrame(img_units)
        if len(centroids) > 1:
            dists = distance.cdist(centroids, centroids)
            np.fill_diagonal(dists, np.inf)
            df_temp['NND_mm'] = np.min(dists, axis=1) * scale
        
        cv2.imwrite(f"processed_{os.path.basename(img_path)}", overlay)
        master_records.append(df_temp)

    return pd.concat(master_records, ignore_index=True)

# Example Usage:
files = [
    'Image (2).jpg',
    'Image (4).jpg'
]
raw_data = extract_raw_lattice_data(files, bridge_gap=3)
raw_data.to_csv("lattice_measurements.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('lattice_measurements.csv')

sns.set_style("whitegrid")

# Plot A: Unit Area Distribution
plt.figure(figsize=(7, 5))
sns.histplot(df['Area_mm2'], kde=True, color='skyblue')
plt.title('Area Distribution ($mm^2$)')
plt.xlabel('Area ($mm^2$)')
plt.tight_layout()
plt.savefig('Plot_A_Area_Distribution.png', dpi=300)
plt.close()

# Plot B: Circularity (Q)
plt.figure(figsize=(7, 5))
sns.histplot(df['Q_Factor'], kde=True, color='salmon')
plt.title('Shape Isomorphism ($Q$)')
plt.xlabel('$Q = 4\pi A / L^2$')
plt.tight_layout()
plt.savefig('Plot_B_Shape_Factor.png', dpi=300)
plt.close()

# Plot C: Spatial Spacing (NND)
plt.figure(figsize=(7, 5))
sns.histplot(df['NND_mm'], kde=True, color='olive')
plt.title('Nearest Neighbor Spacing ($mm$)')
plt.xlabel('Distance ($mm$)')
plt.tight_layout()
plt.savefig('Plot_C_NND_Spacing.png', dpi=300)
plt.close()

# Fig S4E: K-means

In [ ]:
# K-means, with delta values
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd

data = pd.read_csv("delta_all.csv", sep='\t').values.flatten().tolist()
data = np.array([x for x in data if x!=0])

data_reshaped = data.reshape(-1, 1)
for k in range (2, 10):
    kmeans = KMeans(n_clusters=k, random_state=42).fit(data_reshaped)
    print(f"Discrete steps for n={k} :{np.sort(kmeans.cluster_centers_.flatten())}")

# Fig S14: Evans Blue image processing

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def analyze_evans_blue_final_refined(img_path, scale_px_mm=12.0, bias=12, min_area=300):
    img = cv2.imread(img_path)
    if img is None: return None, None, None

    b, g, r = cv2.split(img)
    diff = cv2.subtract(b, r)

    blur = cv2.GaussianBlur(diff, (5, 5), 0)
    denoised = cv2.medianBlur(blur, 5)
    
    otsu_thresh, _ = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    _, mask = cv2.threshold(denoised, otsu_thresh + bias, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    final_cnts = []
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area: continue
            
        hull = cv2.convexHull(c)
        
        x, y, w, h = cv2.boundingRect(hull)
        aspect_ratio = float(w)/h
        if 0.2 < aspect_ratio < 5.0: # Rejects long thin lines
            final_cnts.append(hull)

    results = []
    vis_img = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    for i, cnt in enumerate(final_cnts):
        area_px = cv2.contourArea(cnt)
        M = cv2.moments(cnt)
        if M['m00'] == 0: continue
        cx, cy = int(M['m10']/M['m00']), int(M['m01']/M['m00'])

        m_mask = np.zeros_like(gray)
        cv2.drawContours(m_mask, [cnt], -1, 255, -1)
        mean_val = cv2.mean(gray, mask=m_mask)[0]

        results.append({
            'ID': i + 1,
            'Area_mm2': round(area_px / (scale_px_mm**2), 2),
            'Intensity': round(mean_val, 2)
        })

        # Draw with thicker, smoother lines
        cv2.drawContours(vis_img, [cnt], -1, (0, 255, 0), 2)
        cv2.putText(vis_img, str(i+1), (cx-10, cy+10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    return pd.DataFrame(results), mask, vis_img

In [ ]:
# --- RUN LOOP ---
img_file = 'Image A.tif'
# Try a tighter range. 12 is usually the "sweet spot" for Evans Blue tissue.
BIAS_TESTS = [5, 8, 10, 12, 14] 

fig, axes = plt.subplots(1, len(BIAS_TESTS), figsize=(20, 10))

for i, b in enumerate(BIAS_TESTS):
    df, mask, vis = analyze_evans_blue_final_refined(img_file, bias=b, min_area=400)
    
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Bias: {b} | Spots: {len(df)}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- RUN LOOP ---
img_file = 'Image B.tif'
# Try a tighter range. 12 is usually the "sweet spot" for Evans Blue tissue.
BIAS_TESTS = [5, 8, 10, 12, 14] 

fig, axes = plt.subplots(1, len(BIAS_TESTS), figsize=(20, 10))

for i, b in enumerate(BIAS_TESTS):
    df, mask, vis = analyze_evans_blue_final_refined(img_file, bias=b, min_area=400)
    
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Bias: {b} | Spots: {len(df)}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# --- RUN LOOP ---
img_file = ['Image A.tif', 'Image B.tif']

for img in img_file:
    name = os.path.splitext(os.path.basename(img))[0]
    df, mask, vis = analyze_evans_blue_final_refined(img, bias=8, min_area=400)
    df.to_csv(f"Analyzed_{name}.csv", index=False)
    cv2.imwrite(f"Mask_{name}.png", mask)
    cv2.imwrite(f"Vis_{name}.png", vis)